In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

news_df = pd.read_csv(
    "../MINDsmall_train/news.tsv",
    sep="\t",
    header=None,
    names=[
        "news_id", "category", "subcategory",
        "title", "abstract", "url",
        "title_entities", "abstract_entities"
    ]
)

# Basic cleaning
news_df["title"] = news_df["title"].fillna("")
news_df["abstract"] = news_df["abstract"].fillna("")
news_df["url"] = news_df["url"].fillna("")

news_df = news_df[
    [
        "news_id",
        "category",
        "subcategory",
        "title",
        "abstract",
        "url"
    ]
]

news_df["content"] = news_df["title"] + " " + news_df["abstract"]

print(news_df.shape)
news_df.head()

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(news_df["content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)







# User Profiles 





behaviors_df = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=[
        "impression_id", "user_id", "time",
        "history", "impressions"
    ]
)

behaviors_df["history"] = behaviors_df["history"].fillna("")
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split())

print(behaviors_df.shape)
behaviors_df.head()

news_id_to_index = {
    news_id: idx for idx, news_id in enumerate(news_df["news_id"])
}

from scipy.sparse import vstack
import numpy as np

def build_user_profile(clicked_news_ids):
    vectors = []
    weights = []
    
    total = len(clicked_news_ids)
    
    for i, news_id in enumerate(clicked_news_ids):
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            vectors.append(tfidf_matrix[idx])
            
            # More recent clicks get higher weight
            weights.append((i + 1) / total)
    
    if len(vectors) == 0:
        return None
    
    stacked = vstack(vectors)
    
    weights = np.array(weights).reshape(-1, 1)
    
    weighted_profile = stacked.multiply(weights).sum(axis=0) / weights.sum()
    
    return np.asarray(weighted_profile)

sample_user = behaviors_df.iloc[0]
clicked_articles = sample_user["history"]

print("Sample clicked articles:", clicked_articles[:5])

user_profile_vector = build_user_profile(clicked_articles)

print("User profile shape:", user_profile_vector.shape)







# Cosine Similarity 





similarity_scores = cosine_similarity(user_profile_vector, tfidf_matrix)
similarity_scores = similarity_scores.flatten()

print("Total similarity scores:", len(similarity_scores))






# Generate Top N Recommendations 






def recommend_articles(user_profile, clicked_news_ids, top_n=5):
    
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix)
    similarity_scores = similarity_scores.flatten()
    
    sorted_indices = similarity_scores.argsort()[::-1]
    
    recommendations = []
    
    for idx in sorted_indices:
        news_id = news_df.iloc[idx]["news_id"]
        
        if news_id not in clicked_news_ids:
            recommendations.append(idx)
        
        if len(recommendations) == top_n:
            break
    
    return news_df.iloc[recommendations][
        ["news_id", "title", "abstract", "category", "subcategory", "url"]
    ]

recommendations = recommend_articles(
    user_profile_vector,
    clicked_articles,
    top_n=5
)

recommendations






# Adding Category Filtering






def recommend_articles_filtered(user_profile, clicked_news_ids, 
                                selected_category=None, 
                                selected_subcategory=None,
                                top_n=5):
    
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix)
    similarity_scores = similarity_scores.flatten()
    
    sorted_indices = similarity_scores.argsort()[::-1]
    
    recommendations = []
    
    for idx in sorted_indices:
        row = news_df.iloc[idx]
        
        news_id = row["news_id"]
        
        if news_id in clicked_news_ids:
            continue
        
        if selected_category and row["category"] != selected_category:
            continue
        
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue
        
        recommendations.append(idx)
        
        if len(recommendations) == top_n:
            break
    
    return news_df.iloc[recommendations][
        ["news_id", "title", "abstract", "category", "subcategory", "url"]
    ]

recommendations = recommend_articles_filtered(
    user_profile_vector,
    clicked_articles,
    selected_category="news",
    top_n=5
)

recommendations






# Subcategory Filtering 






news_df["subcategory"].value_counts().head(20)






# Ranking with Transparency 






def recommend_articles_filtered_with_score(user_profile, clicked_news_ids, 
                                           selected_category=None, 
                                           selected_subcategory=None,
                                           top_n=5):
    
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix)
    similarity_scores = similarity_scores.flatten()
    
    sorted_indices = similarity_scores.argsort()[::-1]
    
    recommendations = []
    
    for idx in sorted_indices:
        row = news_df.iloc[idx]
        news_id = row["news_id"]
        
        if news_id in clicked_news_ids:
            continue
        
        if selected_category and row["category"] != selected_category:
            continue
        
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue
        
        recommendations.append((idx, similarity_scores[idx]))
        
        if len(recommendations) == top_n:
            break
    
    results = []
    
    for idx, score in recommendations:
        row = news_df.iloc[idx][
            ["news_id", "title", "abstract", "category", "subcategory", "url"]
        ].to_dict()
        row["similarity_score"] = score
        results.append(row)
    
    return pd.DataFrame(results)

recommendations = recommend_articles_filtered_with_score(
    user_profile_vector,
    clicked_articles,
    selected_category="news",
    selected_subcategory="newspolitics",
    top_n=5
)

recommendations






# Recency Boost 







news_df["recency_score"] = news_df.index / len(news_df)

news_df[["news_id", "recency_score"]].head()

def recommend_articles_with_recency(user_profile, clicked_news_ids, 
                                    selected_category=None, 
                                    selected_subcategory=None,
                                    alpha=0.9,  
                                    beta=0.1,   
                                    top_n=5):
    
    similarity_scores = cosine_similarity(user_profile, tfidf_matrix)
    similarity_scores = similarity_scores.flatten()
    
    sorted_indices = similarity_scores.argsort()[::-1]
    
    recommendations = []
    
    for idx in sorted_indices:
        row = news_df.iloc[idx]
        news_id = row["news_id"]
        
        if news_id in clicked_news_ids:
            continue
        
        if selected_category and row["category"] != selected_category:
            continue
        
        if selected_subcategory and row["subcategory"] != selected_subcategory:
            continue
        
        similarity_score = similarity_scores[idx]
        recency_score = row["recency_score"]
        
        final_score = alpha * similarity_score + beta * recency_score
        
        recommendations.append((idx, similarity_score, recency_score, final_score))
        
        if len(recommendations) >= 100:
            break 
    
    recommendations = sorted(recommendations, key=lambda x: x[3], reverse=True)
    
    recommendations = recommendations[:top_n]
    
    results = []
    
    for idx, sim, rec, final in recommendations:
        row = news_df.iloc[idx][
            ["news_id", "title", "abstract", "category", "subcategory", "url"]
        ].to_dict()
        
        row["similarity_score"] = sim
        row["recency_score"] = rec
        row["final_score"] = final
        
        results.append(row)
    
    return pd.DataFrame(results)

recommendations = recommend_articles_with_recency(
    user_profile_vector,
    clicked_articles,
    selected_category="news",
    selected_subcategory="newspolitics",
    top_n=5
)

recommendations






# Evaluation 

# Precision 





behaviors_eval = pd.read_csv(
    "../MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=[
        "impression_id", "user_id", "time",
        "history", "impressions"
    ]
)

behaviors_eval["history"] = behaviors_eval["history"].fillna("")
behaviors_eval["history"] = behaviors_eval["history"].apply(lambda x: x.split())

def parse_impressions(impression_str):
    result = []
    for item in impression_str.split():
        news_id, label = item.split("-")
        result.append((news_id, int(label)))
    return result

behaviors_eval["impressions"] = behaviors_eval["impressions"].apply(parse_impressions)

behaviors_eval.head()



def precision_at_k_impression(user_profile, impressions, k=5): #single user
    
    candidate_ids = [news_id for news_id, _ in impressions]
    
    scores = []
    
    for news_id in candidate_ids:
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            sim_score = cosine_similarity(
                user_profile, 
                tfidf_matrix[idx]
            )[0][0]
            scores.append((news_id, sim_score))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_k = set([news_id for news_id, _ in scores[:k]])
    
    actual_clicked = set([news_id for news_id, label in impressions if label == 1])
    
    if len(top_k) == 0:
        return 0
    
    hits = len(top_k.intersection(actual_clicked))
    
    return hits / k



sample_row = behaviors_eval.iloc[0]

user_history = sample_row["history"]
impressions = sample_row["impressions"]

user_profile_vector = build_user_profile(user_history)

score = precision_at_k_impression(
    user_profile_vector,
    impressions,
    k=5
)

print("Precision@5 (impression-based):", score)


def evaluate_precision_at_k(num_users=100, k=5): #multiple users
    
    precision_scores = []
    users_evaluated = 0
    
    for i in range(len(behaviors_eval)):
        
        if users_evaluated >= num_users:
            break
        
        row = behaviors_eval.iloc[i]
        
        user_history = row["history"]
        impressions = row["impressions"]
        
        # Skip users with too little history
        if len(user_history) < 3:
            continue
        
        user_profile = build_user_profile(user_history)
        
        if user_profile is None:
            continue
        
        score = precision_at_k_impression(
            user_profile,
            impressions,
            k=k
        )
        
        precision_scores.append(score)
        users_evaluated += 1
    
    average_precision = sum(precision_scores) / len(precision_scores)
    
    print(f"Users evaluated: {users_evaluated}")
    print(f"Average Precision@{k}: {average_precision:.4f}")
    
    return average_precision

evaluate_precision_at_k(num_users=100, k=5)






# Hit rate and MRR






# Hit rate
def hit_rate_at_k_impression(user_profile, impressions, k=5):
    
    candidate_ids = [news_id for news_id, _ in impressions]
    
    scores = []
    
    for news_id in candidate_ids:
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            sim_score = cosine_similarity(
                user_profile, 
                tfidf_matrix[idx]
            )[0][0]
            scores.append((news_id, sim_score))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_k = [news_id for news_id, _ in scores[:k]]
    
    actual_clicked = [news_id for news_id, label in impressions if label == 1]
    
    # Hit = 1 if at least one clicked article is in top K
    for news_id in top_k:
        if news_id in actual_clicked:
            return 1
    
    return 0

# MRR


def mrr_at_k_impression(user_profile, impressions, k=5):
    
    candidate_ids = [news_id for news_id, _ in impressions]
    
    scores = []
    
    for news_id in candidate_ids:
        if news_id in news_id_to_index:
            idx = news_id_to_index[news_id]
            sim_score = cosine_similarity(
                user_profile, 
                tfidf_matrix[idx]
            )[0][0]
            scores.append((news_id, sim_score))
    
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    actual_clicked = [news_id for news_id, label in impressions if label == 1]
    
    for rank, (news_id, _) in enumerate(scores[:k], start=1):
        if news_id in actual_clicked:
            return 1 / rank
    
    return 0

#One user

sample_row = behaviors_eval.iloc[0]

user_history = sample_row["history"]
impressions = sample_row["impressions"]

user_profile_vector = build_user_profile(user_history)

hit_score = hit_rate_at_k_impression(
    user_profile_vector,
    impressions,
    k=5
)

mrr_score = mrr_at_k_impression(
    user_profile_vector,
    impressions,
    k=5
)

print("Hit Rate@5:", hit_score)
print("MRR@5:", mrr_score)

# Multiple users

def evaluate_hit_and_mrr(num_users=100, k=5):
    
    hit_scores = []
    mrr_scores = []
    users_evaluated = 0
    
    for i in range(len(behaviors_eval)):
        
        if users_evaluated >= num_users:
            break
        
        row = behaviors_eval.iloc[i]
        
        user_history = row["history"]
        impressions = row["impressions"]
        
        if len(user_history) < 3:
            continue
        
        user_profile = build_user_profile(user_history)
        
        if user_profile is None:
            continue
        
        hit = hit_rate_at_k_impression(user_profile, impressions, k)
        mrr = mrr_at_k_impression(user_profile, impressions, k)
        
        hit_scores.append(hit)
        mrr_scores.append(mrr)
        
        users_evaluated += 1
    
    avg_hit = sum(hit_scores) / len(hit_scores)
    avg_mrr = sum(mrr_scores) / len(mrr_scores)
    
    print(f"Users evaluated: {users_evaluated}")
    print(f"Average Hit Rate@{k}: {avg_hit:.4f}")
    print(f"Average MRR@{k}: {avg_mrr:.4f}")
    
    return avg_hit, avg_mrr

evaluate_hit_and_mrr(num_users=100, k=5)


# Cold start recommendation



# Show available categories 

print("Available Categories:\n")

categories = sorted(news_df["category"].unique())

for i, cat in enumerate(categories):
    print(f"{i}: {cat}")

# Letting user select category 

cat_index = int(input("\nSelect category number: "))

selected_category = categories[cat_index]

print("Selected Category:", selected_category)

# Show subcategories for selected category

subcats = sorted(
    news_df[news_df["category"] == selected_category]["subcategory"].unique()
)

print("\nAvailable Subcategories:\n")

for i, sub in enumerate(subcats):
    print(f"{i}: {sub}")

# User selects subcatgeory 

sub_index = int(input("\nSelect subcategory number: "))

selected_subcategory = subcats[sub_index]

print("Selected Subcategory:", selected_subcategory)

# Create a sample user

def create_user_profile_from_interest(category, subcategory, n_seed=5):
    
    seed_articles = news_df[
        (news_df["category"] == category) &
        (news_df["subcategory"] == subcategory)
    ].head(n_seed)
    
    clicked_ids = seed_articles["news_id"].tolist()
    
    user_profile = build_user_profile(clicked_ids)
    
    return user_profile, clicked_ids

# Recommendation 

user_profile, clicked_ids = create_user_profile_from_interest(
    selected_category,
    selected_subcategory
)

recommendations = recommend_articles_with_recency(
    user_profile,
    clicked_ids,
    selected_category=selected_category,
    selected_subcategory=selected_subcategory,
    top_n=5
)
if recommendations.empty:
    
    print("Not enough articles in this subcategory. Falling back to category level...\n")
    
    recommendations = recommend_articles_with_recency(
        user_profile,
        clicked_ids,
        selected_category=selected_category,
        selected_subcategory=None,
        top_n=5
    )

recommendations










(51282, 7)
TF-IDF Matrix Shape: (51282, 5000)
(156965, 5)
Sample clicked articles: ['N55189', 'N42782', 'N34694', 'N45794', 'N18445']
User profile shape: (1, 5000)
Total similarity scores: 51282
Precision@5 (impression-based): 0.2
Users evaluated: 100
Average Precision@5: 0.1160
Hit Rate@5: 1
MRR@5: 1.0
Users evaluated: 100
Average Hit Rate@5: 0.5300
Average MRR@5: 0.3327
Available Categories:

0: autos
1: entertainment
2: finance
3: foodanddrink
4: health
5: kids
6: lifestyle
7: middleeast
8: movies
9: music
10: news
11: northamerica
12: sports
13: travel
14: tv
15: video
16: weather
Selected Category: news

Available Subcategories:

0: causes
1: causes-disaster-relief
2: causes-environment
3: causes-military-appreciation
4: causes-poverty
5: elections-2020-us
6: empowering-the-planet
7: factcheck
8: indepth
9: narendramodi_opinion
10: newsbusiness
11: newscrime
12: newselection2020
13: newsfactcheck
14: newsgoodnews
15: newsnational
16: newsoffbeat
17: newsopinion
18: newsother
19: n

,news_id,title,abstract,category,subcategory,url,similarity_score,recency_score,final_score
0,N33983,In Photos: Veterans Day,,news,newsus,https://assets.msn.com/labs/mind/BBWyu1r.html,0.415187,0.995203,0.473188
1,N64968,Photos of the Day,Our top photos from the last 24 hours.,news,newsworld,https://assets.msn.com/labs/mind/BBWHmtN.html,0.430517,0.817421,0.469207
2,N25160,Photos of the Day,Our top photos from the last 24 hours.,news,newsworld,https://assets.msn.com/labs/mind/BBWKBfG.html,0.430517,0.673336,0.454799
3,N16662,Photos of the Day,Our top photos from the last 24 hours.,news,newsworld,https://assets.msn.com/labs/mind/AAJw8jm.html,0.430517,0.564779,0.443943
4,N43450,Photos of the Day,Our top photos from the last 24 hours.,news,newsworld,https://assets.msn.com/labs/mind/AAJzxtw.html,0.430517,0.521177,0.439583
